In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import models, datasets, transforms


In [ ]:
# kaggle에서 이미지 다운
!mkdir -p /content/stanford_dogs

!kaggle datasets download \
  -d jessicali9530/stanford-dogs-dataset \
  -p /content/stanford_dogs

Dataset URL: https://www.kaggle.com/datasets/jessicali9530/stanford-dogs-dataset
License(s): other
100% 750M/750M [00:21<00:00, 36.6MB/s]



In [ ]:
# 다운로드 된 이미지 .zip파일 압축해제
import os
import zipfile

download_dir = "/content/stanford_dogs"

zip_files = [
    f for f in os.listdir(download_dir)
    if f.endswith(".zip")
]

print("zip 파일:", zip_files)

for zip_file in zip_files:
    zip_path = os.path.join(download_dir, zip_file)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(download_dir)

print("zip 압축 해제 완료")

zip 파일: ['stanford-dogs-dataset.zip']
zip 압축 해제 완료


In [ ]:
# stanford-dogs-dataset에서 10가지 품종만 추출해서 새로운 폴더로 저장
import os
import shutil

source_dir = "/content/stanford_dogs/images/Images"
target_dir = "/content/dog_breeds_10"

selected_keywords = [
    "Chihuahua",
    "Maltese",
    "beagle",
    "pug",
    "golden_retriever",
    "Labrador_retriever",
    "Siberian_husky",
    "Pomeranian",
    "toy_poodle",
    "Samoyed"
]

# 기존 10품종 폴더가 있으면 삭제 후 다시 생성
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)

os.makedirs(target_dir, exist_ok=True)

all_folders = os.listdir(source_dir)
selected_folders = []

for keyword in selected_keywords:
    matched = None

    for folder in all_folders:
        if keyword.lower() in folder.lower():
            matched = folder
            break

    if matched is not None:
        selected_folders.append(matched)
    else:
        print("못 찾은 품종:", keyword)

print("선택된 품종 폴더:")
for folder in selected_folders:
    print(folder)

# 선택된 품종 폴더만 복사
for folder in selected_folders:
    src = os.path.join(source_dir, folder)
    dst = os.path.join(target_dir, folder)
    shutil.copytree(src, dst)

print("복사 완료")
print("선택된 품종 개수:", len(os.listdir(target_dir)))

선택된 품종 폴더:
n02085620-Chihuahua
n02085936-Maltese_dog
n02088364-beagle
n02110958-pug
n02099601-golden_retriever
n02099712-Labrador_retriever
n02110185-Siberian_husky
n02112018-Pomeranian
n02113624-toy_poodle
n02111889-Samoyed
복사 완료
선택된 품종 개수: 10


In [ ]:
# 5주차 위클리 챌린지 1번
# 1.ResNet 모델을 불러와 새로운 이미지 데이터셋을 분류하세요.

# 가상 데이터셋 생성
num_classes = 10
input_shape = (3, 244, 244)

x_train = np.random.random((1000, 3, 244, 244)).astype(np.float32)
y_train = np.random.randint(num_classes, size=(1000,))

x_test = np.random.random((200, 3, 244, 244)).astype(np.float32)
y_test = np.random.randint(num_classes, size=(200,))


train_dataset = TensorDataset(torch.tensor(x_train), torch.tensor(y_train))
test_dataset = TensorDataset(torch.tensor(x_test), torch.tensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# ResNet50 모델 로드
base_model = models.resnet50(weights=None)
base_model = nn.Sequential(*list(base_model.children())[:-2])


# 새로운 모델 구축
class CustomResNet50(nn.Module):
  def __init__(self, num_classes):
    super(CustomResNet50, self).__init__()
    self.base_model = base_model
    self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc1 = nn.Linear(2048, 256)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(256, num_classes)
    self.softmax = nn.Softmax(dim=1)

  def forward(self, x):
    x = self.base_model(x)
    x = self.global_avg_pool(x)
    x = torch.flatten(x, 1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    x = self.softmax(x)
    return x


model = CustomResNet50(num_classes = num_classes)

# 손실함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.0001)


# 모델 훈련
num_epochs = 10

for epoch in range(num_epochs) :
  model.train()
  running_loss = 0.0
  for inputs, labels in train_loader :
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')


# 모델 평가
model.eval()
correct = 0
total = 0
with torch.no_grad() :
  for inputs, labels in test_loader :
    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')



Epoch 1, Loss: 2.3065835684537888
Epoch 2, Loss: 2.2917394563555717
Epoch 3, Loss: 2.219032183289528
Epoch 4, Loss: 2.156795047223568
Epoch 5, Loss: 2.100365374237299
Epoch 6, Loss: 2.086160857230425
Epoch 7, Loss: 2.064759898930788
Epoch 8, Loss: 2.060050528496504
Epoch 9, Loss: 2.0212524943053722
Epoch 10, Loss: 2.0221326872706413
Test Accuracy: 5.00%


nn.Sequential과 list(base_model.children())[:-2]을 사용하여 ResNet50 모델의 마지막 두 레이어를 제거합니다. 이는 전역 평균 풀링(Global Average Pooling) 레이어를 추가하기 위해서입니다.

이 내용을 자세히 학습하기 위해서 GPT에게 물어봤습니다

GPT 답 :

In [ ]:
# 위클리 챌린지 1번 (캐글에서 강아지 품종 데이터 가져온 버젼)

# 강아지 이미지 폴더 경로
data_dir = "/content/dog_breeds_10"

# 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((244, 244)),
    transforms.ToTensor()
])

# 실제 강아지 이미지 데이터셋 불러오기
full_dataset = datasets.ImageFolder(
    root=data_dir,
    transform=transform
)

# 클래스 개수 자동 설정
num_classes = len(full_dataset.classes)

# train/test 분리
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(full_dataset,[train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# ResNet50 모델 로드
base_model = models.resnet50(weights=None)
base_model = nn.Sequential(*list(base_model.children())[:-2])


# 새로운 모델 구축
class CustomResNet50(nn.Module):
  def __init__(self, num_classes):
    super(CustomResNet50, self).__init__()
    self.base_model = base_model
    self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc1 = nn.Linear(2048, 256)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(256, num_classes)

  def forward(self, x):
    x = self.base_model(x)
    x = self.global_avg_pool(x)
    x = torch.flatten(x, 1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    return x


model = CustomResNet50(num_classes = num_classes)

# 손실함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.0001)


# 모델 훈련
num_epochs = 10

for epoch in range(num_epochs) :
  model.train()
  running_loss = 0.0
  for inputs, labels in train_loader :
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')


# 모델 평가
model.eval()
correct = 0
total = 0
with torch.no_grad() :
  for inputs, labels in test_loader :
    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')


Epoch 1, Loss: 2.295677954951922
Epoch 2, Loss: 2.2367088298002877
Epoch 3, Loss: 2.1560738955934844
Epoch 4, Loss: 2.1098197773098946
Epoch 5, Loss: 2.0519977509975433
Epoch 6, Loss: 2.0170688604315123
Epoch 7, Loss: 1.9957696422934532
Epoch 8, Loss: 1.952751783033212
Epoch 9, Loss: 1.9108175685008366
Epoch 10, Loss: 1.8739399760961533
Test Accuracy: 21.32%


강아지 품종 데이터셋을 불러오는 과정을 AI에게 질문해서 이해한 다음 작성하였습니다.

또한 코드 리뷰를 했을 때 CrossEntropyLoss()에 softmax가 포함 되어있어 제거 하는 것이 낫다고 해서 제거했습니다.

In [ ]:
# 5주차 위클리 챌린지 2번 (VGG16 모델만 사용)
# 2. 이미지 데이터셋과 사전 훈련된 VGG16 모델을 가져와 전이학습을 수행하세요.

num_classes = 10
input_shape = (3, 244, 244)

x_train = np.random.random((1000, 3, 244, 244)).astype(np.float32)
y_train = np.random.randint(num_classes, size=(1000,))

x_test = np.random.random((200, 3, 244, 244)).astype(np.float32)
y_test = np.random.randint(num_classes, size=(200,))


train_dataset = TensorDataset(torch.tensor(x_train), torch.tensor(y_train))
test_dataset = TensorDataset(torch.tensor(x_test), torch.tensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

base_model = models.vgg16(weights=None)
base_model.classifier = nn.Sequential(*list(base_model.classifier.children())[:-1])

class CustomVGG16(nn.Module) :
  def __init__(self, num_classes) :
    super(CustomVGG16, self).__init__()
    self.base_model = base_model
    self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))
    self.fc1 = nn.Linear(512, 256)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(256, num_classes)

  def forward(self, x) :
    x = self.base_model.features(x)
    x = self.global_avg_pool(x)
    x = torch.flatten(x, 1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    return x

model = CustomVGG16(num_classes = num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.0001)

num_epoch = 10

for epoch in range(num_epoch) :
  model.train()
  running_loss = 0
  for inputs, labels in train_loader :
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')


model.eval()
correct = 0
total = 0
with torch.no_grad() :
  for inputs, labels in test_loader :
    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()


accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')


Epoch 1, Loss: 2.304735019803047
Epoch 2, Loss: 2.30266036093235
Epoch 3, Loss: 2.30257960408926
Epoch 4, Loss: 2.301813267171383
Epoch 5, Loss: 2.301850825548172
Epoch 6, Loss: 2.3018133491277695
Epoch 7, Loss: 2.3000457733869553
Epoch 8, Loss: 2.303008057177067
Epoch 9, Loss: 2.301477760076523
Epoch 10, Loss: 2.3009190261363983
Test Accuracy: 7.50%


전이 학습을 할 때 Fine Tuning, Feature Extraction 중 뭐가 더 나을지 AI에게 물어보고 진행하려고 했습니다.

AI 답 : Feature Extraction은 사전 훈련된 모델의 합성곱 계층을 고정하고 마지막 분류 계층만 새 데이터셋에 맞게 학습하는 방식이다.
반면 Fine-tuning은 사전 훈련된 모델의 일부 계층까지 함께 학습시켜 새 데이터셋에 더 적합하도록 조정하는 방식이다.

Fine-tuning은 성능이 더 좋아질 가능성이 있지만, 데이터가 적으면 과적합 위험이 있고 학습 시간이 오래 걸린다.
따라서 본 실습에서는 기본적인 전이학습 방식인 Feature Extraction을 먼저 적용하고, 성능 개선이 필요할 경우 마지막 합성곱 블록을 추가로 학습하는 Fine-tuning을 고려할 수 있다.

결론 : 결국 Feature Extraction을 한 후 Fine Tuning을 진행해야 더 많이 성능개선이 된다고 해서 둘 다 진행했습니다.

In [ ]:
# 위클리 챌린지 2번 (전이학습 진행)

# 이미지 폴더로 데이터셋 불러오기 (AI)
data_dir = "/content/dog_breeds_10"

transform = transforms.Compose([
    transforms.Resize((244, 244)),
    transforms.ToTensor(),

    # VGG16은 ImageNet으로 사전 학습되었기 때문에 ImageNet 정규화 사용
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_dataset = datasets.ImageFolder(
    root=data_dir,
    transform=transform
)

num_classes = len(full_dataset.classes)

# train, test 데이터셋 분류
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size


train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

base_model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

# 모델 정의
class CustomVGG16(nn.Module) :
  def __init__(self, num_classes) :
    super(CustomVGG16, self).__init__()
    self.base_model = base_model.features
    self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))
    self.fc1 = nn.Linear(512, 256)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(256, num_classes)

  def forward(self, x) :
    x = self.base_model(x)
    x = self.global_avg_pool(x)
    x = torch.flatten(x, 1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)

    return x

model = CustomVGG16(num_classes=num_classes)

# Feture Extraction 설정
# VGG16 feature extractor 고정
for param in model.base_model.parameters():
    param.requires_grad = False


criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

# Feature Extraction 학습
num_epoch = 5

for epoch in range(num_epoch) :
  model.train()
  running_loss = 0
  for inputs, labels in train_loader :
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

# Feature Extraction 평가(중간 평가)
model.eval()
correct = 0
total = 0
with torch.no_grad() :
  for inputs, labels in test_loader :
    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()


accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

# Fine Tuning 설정
# VGG16 마지막 convolution block 일부만 학습 가능하게 변경
for param in model.base_model[24:].parameters():
    param.requires_grad = True

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

# Fine Tuning 학습
fine_tuning_epochs = 5

for epoch in range(fine_tuning_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f'Fine-tuning Epoch {epoch + 1}, Loss: {running_loss / len(train_loader):.4f}')

# Fine Tuning까지 최종 평가
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f'Fine-tuning Test Accuracy: {accuracy * 100:.2f}%')

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 225MB/s]


Epoch 1, Loss: 1.8698969557881355
Epoch 2, Loss: 1.5498311966657639
Epoch 3, Loss: 1.5152409573396046
Epoch 4, Loss: 1.501858909924825
Epoch 5, Loss: 1.491685616473357
Test Accuracy: 95.26%
Fine-tuning Epoch 1, Loss: 1.5038
Fine-tuning Epoch 2, Loss: 1.4825
Fine-tuning Epoch 3, Loss: 1.4827
Fine-tuning Epoch 4, Loss: 1.4759
Fine-tuning Epoch 5, Loss: 1.4690
Fine-tuning Test Accuracy: 94.47%


교재에는 Fine Tuning을 할 때 데이터셋을 train, val로 나누어서 진행했습니다. 이 부분이 이해가 잘 안되어서 AI에게 질문햇습니다.

AI 답 : 왜 train/val로 나누는 게 낫냐?

모델은 train 데이터로 학습하고, val 데이터로 학습 중간 성능을 확인해야 해.

그래야 모델이 진짜로 잘 배우는지, 아니면 train 데이터만 외우는지 확인할 수 있어.

검증 데이터에도 랜덤 증강이 적용돼버릴 수 있어.

그러면 평가할 때마다 이미지가 다르게 변해서 검증 결과가 불안정해질 수 있어.



작성 후 GPT에게 코드리뷰를 해달라고 얘기 했습니다.

이후 수정사항은 오타 수정과, Feature Extraction 고정 대상

for param in base_model.parameters():   
    param.requires_grad = False

CustomVGG16 안에 self.base_model = base_model.features 로 정의되어 있어서 이 부분을

for param in model.base_model.parameters():
    param.requires_grad = False

로 변경하는게 낫다고 합니다.

또한 Fine Tuning 학습률은 더 낮추는 게 좋음

현재:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

Fine Tuning에서는 기존 VGG16 가중치를 살짝만 수정해야 해서 보통 더 낮게 잡아.

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.00001)

이게 더 안전해.

그래서 Feature Extraction 부분은 0.001로 Fine Tuning 부분은 0.0001로 변경하였습니다.

In [ ]:
# 위클리 챌린지 3번
# 동일한 데이터셋에서 ResNet과 VGG16을 각각 학습시켜 성능을 비교하세요.

# 이미지 폴더로 데이터셋 불러오기 (AI)
data_dir = "/content/dog_breeds_10"

transform = transforms.Compose([
    transforms.Resize((244, 244)),
    transforms.ToTensor(),

    # VGG16은 ImageNet으로 사전 학습되었기 때문에 ImageNet 정규화 사용
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_dataset = datasets.ImageFolder(
    root=data_dir,
    transform=transform
)

num_classes = len(full_dataset.classes)

# train, test 데이터셋 분류
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size


train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ResNet50 모델 로드
base_model = models.resnet50(weights=None)
base_model = nn.Sequential(*list(base_model.children())[:-2])

# ResNet 모델 정의
class CustomResNet50(nn.Module) :
  def __init__(self, num_classes):
    super(CustomResNet50, self).__init__()
    self.base_model = base_model
    self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))
    self.fc1 = nn.Linear(2048, 256)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(256, num_classes)

  def forward(self, x):
    x = self.base_model(x)
    x = self.global_avg_pool(x)
    x = torch.flatten(x, 1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)

    return x

model = CustomResNet50(num_classes=num_classes)

# 손실함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# CustomResNet50 모델 학습
num_epochs = 10

for epoch in range(num_epochs) :
  model.train()
  running_loss = 0.0
  for inputs, labels in train_loader:
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  print(f"Epoch: {epoch+1}, CustomResNet50 Loss: {running_loss/len(train_loader)}")

# 모델 평가
model.eval()
correct = 0
total = 0
with torch.no_grad() :
  for inputs, labels in test_loader:
    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()


accuracy = correct / total
print(f'CustomResNet50 Accuracy: {accuracy * 100:.2f}%')


# VGG16 모델 로드
base_model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

# CustomVGG16 모델 정의
class CustomVGG16(nn.Module) :
  def __init__(self, num_classes) :
    super(CustomVGG16, self).__init__()
    self.base_model = base_model.features
    self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))
    self.fc1 = nn.Linear(512, 256)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(256, num_classes)

  def forward(self, x):
    x = self.base_model(x)
    x = self.global_avg_pool(x)
    x = torch.flatten(x, 1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)

    return x

model = CustomVGG16(num_classes=num_classes)

# VGG16 feature extractor 고정
for param in model.base_model.parameters():
    param.requires_grad = False


criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

# 모델 학습
num_epochs = 10

for epoch in range(num_epochs) :
  model.train()
  running_loss = 0.0
  for inputs, labels in train_loader :
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  print(f"Epoch : {epoch + 1},  CustomVGG16 Loss : {running_loss / len(train_loader)}")

# 모델 평가
model.eval()
correct = 0
total = 0

with torch.no_grad() :
  for inputs, labels in test_loader :
    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f"CustomVGG16 Accuracy : {accuracy * 100:.2f}%")


Epoch: 1, CustomResNet50 Loss: 2.3149439245462418
Epoch: 2, CustomResNet50 Loss: 2.248255640268326
Epoch: 3, CustomResNet50 Loss: 2.1787365774313607
Epoch: 4, CustomResNet50 Loss: 2.1457239761948586
Epoch: 5, CustomResNet50 Loss: 2.0841964011391005
Epoch: 6, CustomResNet50 Loss: 2.034128300845623
Epoch: 7, CustomResNet50 Loss: 1.9976146817207336
Epoch: 8, CustomResNet50 Loss: 1.8962185929218929
Epoch: 9, CustomResNet50 Loss: 1.8480523750185966
Epoch: 10, CustomResNet50 Loss: 1.7733080983161926
CustomResNet50 Accuracy: 21.84%
Epoch : 1,  CustomVGG16 Loss : 0.8137877136468887
Epoch : 2,  CustomVGG16 Loss : 0.1737586814754953
Epoch : 3,  CustomVGG16 Loss : 0.11879926733672619
Epoch : 4,  CustomVGG16 Loss : 0.08421462611295283
Epoch : 5,  CustomVGG16 Loss : 0.05723501767109459
Epoch : 6,  CustomVGG16 Loss : 0.039198551375496514
Epoch : 7,  CustomVGG16 Loss : 0.028434562550197977
Epoch : 8,  CustomVGG16 Loss : 0.02304246307661136
Epoch : 9,  CustomVGG16 Loss : 0.01774427438310037
Epoch : 10

In [ ]:
# 위클리 챌린지 4번
# 가상 데이터셋을 생성한 뒤, GridSearch와 RandomSearch 기법으로 하이퍼파라미터 튜닝을 진행하세요.

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from scipy.stats import uniform
from sklearn.model_selection import ParameterSampler
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# GridSearch 기법
# 데이터셋 생성
num_classes = 10
input_shape = (32, 32, 3)
num_sample = 1000

x = np.random.rand(num_sample, *input_shape)
y = np.random.randint(num_classes, size=num_sample)

x, y = shuffle(x,y)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 모델 정의
def create_model(learning_rate=0.001):
  base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
  x = base_model.output
  x = GlobalAveragePooling2D()(x)
  x = Dense(1024, activation='relu')(x)
  prediction = Dense(num_classes, activation='softmax')(x)

  model = Model(inputs=base_model.input, outputs = prediction)
  opt = Adam(learning_rate=learning_rate)

  model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

  return model

# 모델 훈련 및 평가
learning_rates = [0.001, 0.01, 0.1]
batch_sizes = [16, 32, 64]
epochs = 10

best_accuracy = 0
best_params = {}

for lr in learning_rates :
  for batch_size in batch_sizes :
    model = create_model(learning_rate=lr)
    model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.2, verbose=1)

    # 모델 평가
    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
    print(f"Learning Rate: {lr}, Batch Size: {batch_size}, Test Accuracy: {accuracy}")

    if accuracy > best_accuracy :
      best_accuracy = accuracy
      best_params = {'learning_rate': lr, 'batch_size': batch_size}

print(f"Best Accuracy: {best_accuracy}, Best Parameters: {best_params}")

# 최적의 하이퍼파라미터로 모델 재학습
epochs = 10
best_model = create_model(learning_rate=best_params["learning_rate"])
best_model.fit(x_train, y_train, batch_size=best_params["batch_size"], epochs=epochs, validation_split=0.2)

# 최적의 하이퍼파라미터 모델 평가
test_loss, test_accuracy = best_model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_accuracy:.2f}")
print(f"Test Loss: {test_loss:.2f}")



# RandomSearch 기법
# 더 오래 기억하기 위해 데이터셋 생성부터 다시하겠습니다.
num_classes = 10
input_shape = (32, 32, 3)
num_sample = 1000

x = np.random.rand(num_sample, *input_shape)
y = np.random.randint(num_classes, size=num_sample)

x, y = shuffle(x,y)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 모델 생성 함수
def create_model(learning_rate=0.001):
  base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
  x = base_model.output
  x = GlobalAveragePooling2D()(x)
  x = Dense(1024, activation='relu')(x)
  predictions = Dense(num_classes, activation='softmax')(x)

  model = Model(inputs=base_model.input, outputs=predictions)
  opt = Adam(learning_rate=learning_rate)

  model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=["accuracy"])
  return model

# 하이퍼파라미터 분포 정의
param_dist = {
    'learning_rate': uniform(0.001, 0.1),
    'batch_size' : [16, 32, 64]
}

best_accuracy = 0
best_params = {}
n_iter = 5

# 랜덤하게 하이퍼파라미터 조합을 선택하여 시도
for params in ParameterSampler(param_dist, n_iter=n_iter, random_state=42):
  model = create_model(learning_rate=params['learning_rate'])
  model.fit(x_train, y_train, batch_size=params['batch_size'], epochs=epochs, validation_split=0.2, verbose=0)

  # 모델 평가
  loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
  print(f"Learning Rate: {params['learning_rate']}, Batch Size: {params['batch_size']}, Test Accuracy: {accuracy}")

  if accuracy > best_accuracy :
    best_accuracy = accuracy
    best_params = params

print(f"Best Accuracy: {best_accuracy}, Best Parameters: {best_params}")

# 최적 하이퍼파라미터로 모델 재학습
epochs = 10
best_model = create_model(learning_rate=best_params['learning_rate'])
best_model.fit(x_train, y_train, epochs=epochs, batch_size=best_params['batch_size'], validation_split=0.2)

# 테스트 데이터셋으로 모델 평가
test_loss, test_accuracy = best_model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_accuracy:.2f}")
print(f"Test Loss: {test_loss:.2f}")

Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 63s 148ms/step - accuracy: 0.0891 - loss: 3.4920 - val_accuracy: 0.0938 - val_loss: 219.9027
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.1484 - loss: 2.4822 - val_accuracy: 0.1250 - val_loss: 323.0866
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.2000 - loss: 2.3037 - val_accuracy: 0.0875 - val_loss: 26.3495
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.2937 - loss: 2.0691 - val_accuracy: 0.1063 - val_loss: 23.0575
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.3734 - loss: 1.8829 - val_accuracy: 0.0875 - val_loss: 42.9370
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.4250 - loss: 1.6977 - val_accuracy: 0.0875 - val_loss: 9.2692
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.4875 - loss: 1.5417 - val_accuracy: 0.0812 - val_loss: 2.3027
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.5188 - loss: 1.4471 - val_accuracy: 0